# Heathrow Financial Model and Variances
## 03 — Revenue, Operating Costs and Scenarios

### Purpose
Connect passenger activity to revenue and operating-cost assumptions
to build a transparent 2026 financial outlook.

### Data labels
- Reported actuals: published annual financial results and monthly passengers.
- Model assumptions: revenue yields, cost behaviour and baseline choices.
- Forecast outputs: estimated revenue, costs and adjusted EBITDA.

### Important limitations
Monthly financial estimates are not Heathrow's reported management accounts.
Our model baseline is not Heathrow's approved internal budget.
Capital expenditure and financing costs are excluded from adjusted EBITDA.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

financial_annual = pd.read_csv(
    PROCESSED_DIR / "heathrow_financial_annual.csv"
)

financial_annual[
    [
        "year",
        "revenue_gbp_m",
        "adjusted_operating_costs_gbp_m",
        "adjusted_ebitda_gbp_m"
    ]
]

,year,revenue_gbp_m,adjusted_operating_costs_gbp_m,adjusted_ebitda_gbp_m
0,2024,3559,1524,2035
1,2025,3623,1589,2034


In [3]:
traffic_monthly = pd.read_csv(
    PROCESSED_DIR / "heathrow_traffic_monthly.csv",
    parse_dates=["month"]
)

passenger_forecast = pd.read_csv(
    PROCESSED_DIR / "heathrow_passenger_forecast_2026_q4.csv",
    parse_dates=["month"]
)

print("Reported traffic rows:", len(traffic_monthly))
print("Q4 forecast rows:", len(passenger_forecast))

Reported traffic rows: 24
Q4 forecast rows: 3


In [4]:
actual_2026 = (
    traffic_monthly.loc[
        traffic_monthly["month"].dt.year == 2026,
        ["month", "passengers"]
    ]
    .assign(passenger_status="reported_actual")
)

forecast_2026 = (
    passenger_forecast[["month", "forecast_passengers"]]
    .rename(columns={"forecast_passengers": "passengers"})
    .assign(passenger_status="model_forecast")
)

model_2026 = (
    pd.concat([actual_2026, forecast_2026], ignore_index=True)
    .sort_values("month")
    .reset_index(drop=True)
)

model_2026

,month,passengers,passenger_status
0,2026-01-01,6460877,reported_actual
1,2026-02-01,5819080,reported_actual
2,2026-03-01,6645184,reported_actual
3,2026-04-01,6721032,reported_actual
4,2026-05-01,7121212,reported_actual
5,2026-06-01,7227494,reported_actual
6,2026-07-01,7859408,reported_actual
7,2026-08-01,7866510,reported_actual
8,2026-09-01,7179707,reported_actual
9,2026-10-01,7365772,model_forecast


In [5]:
expected_months = pd.date_range(
    "2026-01-01", "2026-12-01", freq="MS"
)

print("Rows:", len(model_2026))
print("Duplicate months:", model_2026["month"].duplicated().sum())
print("Missing months:", expected_months.difference(model_2026["month"]).tolist())
print("Missing passenger values:", model_2026["passengers"].isna().sum())
print(f"Full-year passengers: {model_2026['passengers'].sum():,.0f}")

Rows: 12
Duplicate months: 0
Missing months: []
Missing passenger values: 0
Full-year passengers: 84,088,953


In [6]:
benchmark_2025 = financial_annual.set_index("year").loc[2025]

revenue_per_passenger_assumption = (
    benchmark_2025["revenue_gbp_m"] * 1_000_000
    / benchmark_2025["passengers"]
)

print(
    "Assumed revenue per passenger:",
    f"£{revenue_per_passenger_assumption:.2f}"
)

Assumed revenue per passenger: £42.89


In [7]:
model_2026["revenue_per_passenger_assumption_gbp"] = (
    revenue_per_passenger_assumption
)

model_2026["modelled_revenue_gbp_m"] = (
    model_2026["passengers"]
    * revenue_per_passenger_assumption
    / 1_000_000
)

model_2026[
    ["month", "passenger_status", "modelled_revenue_gbp_m"]
].round({"modelled_revenue_gbp_m": 2})

,month,passenger_status,modelled_revenue_gbp_m
0,2026-01-01,reported_actual,277.14
1,2026-02-01,reported_actual,249.61
2,2026-03-01,reported_actual,285.04
3,2026-04-01,reported_actual,288.30
4,2026-05-01,reported_actual,305.46
5,2026-06-01,reported_actual,310.02
6,2026-07-01,reported_actual,337.13
7,2026-08-01,reported_actual,337.43
8,2026-09-01,reported_actual,307.97
9,2026-10-01,model_forecast,315.95


In [8]:
fixed_cost_share = 0.70
variable_cost_share = 0.30

annual_cost_benchmark_gbp = (
    benchmark_2025["adjusted_operating_costs_gbp_m"] * 1_000_000
)

monthly_fixed_cost_gbp = (
    annual_cost_benchmark_gbp * fixed_cost_share / 12
)

variable_cost_per_passenger_gbp = (
    annual_cost_benchmark_gbp * variable_cost_share
    / benchmark_2025["passengers"]
)

print(f"Monthly fixed costs: £{monthly_fixed_cost_gbp:,.0f}")
print(
    "Variable cost per passenger:",
    f"£{variable_cost_per_passenger_gbp:.2f}"
)

Monthly fixed costs: £92,691,667
Variable cost per passenger: £5.64


In [9]:
model_2026["modelled_fixed_cost_gbp_m"] = (
    monthly_fixed_cost_gbp / 1_000_000
)

model_2026["modelled_variable_cost_gbp_m"] = (
    model_2026["passengers"]
    * variable_cost_per_passenger_gbp
    / 1_000_000
)

model_2026["modelled_operating_cost_gbp_m"] = (
    model_2026["modelled_fixed_cost_gbp_m"]
    + model_2026["modelled_variable_cost_gbp_m"]
)

model_2026[
    [
        "month",
        "modelled_fixed_cost_gbp_m",
        "modelled_variable_cost_gbp_m",
        "modelled_operating_cost_gbp_m"
    ]
].round({
    "modelled_fixed_cost_gbp_m": 2,
    "modelled_variable_cost_gbp_m": 2,
    "modelled_operating_cost_gbp_m": 2
})

,month,modelled_fixed_cost_gbp_m,modelled_variable_cost_gbp_m,modelled_operating_cost_gbp_m
0,2026-01-01,92.69,36.46,129.16
1,2026-02-01,92.69,32.84,125.53
2,2026-03-01,92.69,37.50,130.20
3,2026-04-01,92.69,37.93,130.62
4,2026-05-01,92.69,40.19,132.88
5,2026-06-01,92.69,40.79,133.48
6,2026-07-01,92.69,44.36,137.05
7,2026-08-01,92.69,44.40,137.09
8,2026-09-01,92.69,40.52,133.21
9,2026-10-01,92.69,41.57,134.26


In [10]:
model_2026["modelled_adjusted_ebitda_gbp_m"] = (
    model_2026["modelled_revenue_gbp_m"]
    - model_2026["modelled_operating_cost_gbp_m"]
)

model_2026["modelled_adjusted_ebitda_margin_pct"] = (
    model_2026["modelled_adjusted_ebitda_gbp_m"]
    / model_2026["modelled_revenue_gbp_m"]
    * 100
)

model_2026[
    [
        "month",
        "modelled_adjusted_ebitda_gbp_m",
        "modelled_adjusted_ebitda_margin_pct"
    ]
].round({
    "modelled_adjusted_ebitda_gbp_m": 2,
    "modelled_adjusted_ebitda_margin_pct": 2
})

,month,modelled_adjusted_ebitda_gbp_m,modelled_adjusted_ebitda_margin_pct
0,2026-01-01,147.98,53.40
1,2026-02-01,124.07,49.71
2,2026-03-01,154.85,54.32
3,2026-04-01,157.67,54.69
4,2026-05-01,172.58,56.50
5,2026-06-01,176.54,56.94
6,2026-07-01,200.08,59.35
7,2026-08-01,200.34,59.37
8,2026-09-01,174.76,56.74
9,2026-10-01,181.69,57.51


In [11]:
annual_revenue = model_2026["modelled_revenue_gbp_m"].sum()
annual_costs = model_2026["modelled_operating_cost_gbp_m"].sum()
annual_ebitda = model_2026["modelled_adjusted_ebitda_gbp_m"].sum()

annual_margin = annual_ebitda / annual_revenue * 100

print(f"Modelled 2026 revenue: £{annual_revenue:,.2f}m")
print(f"Modelled 2026 operating costs: £{annual_costs:,.2f}m")
print(f"Modelled 2026 adjusted EBITDA: £{annual_ebitda:,.2f}m")
print(f"Modelled 2026 adjusted EBITDA margin: {annual_margin:.2f}%")

Modelled 2026 revenue: £3,606.95m
Modelled 2026 operating costs: £1,586.89m
Modelled 2026 adjusted EBITDA: £2,020.06m
Modelled 2026 adjusted EBITDA margin: 56.00%


In [12]:
reconstructed_revenue = (
    benchmark_2025["passengers"]
    * revenue_per_passenger_assumption
    / 1_000_000
)

reconstructed_costs = (
    monthly_fixed_cost_gbp * 12
    + benchmark_2025["passengers"]
    * variable_cost_per_passenger_gbp
) / 1_000_000

print(
    "2025 revenue difference (£m):",
    round(reconstructed_revenue - benchmark_2025["revenue_gbp_m"], 8)
)

print(
    "2025 cost difference (£m):",
    round(
        reconstructed_costs
        - benchmark_2025["adjusted_operating_costs_gbp_m"],
        8
    )
)

2025 revenue difference (£m): 0.0
2025 cost difference (£m): 0.0


### Base financial-model assumptions

- Revenue per passenger stays at the 2025 annual average.
- 70% of 2025 adjusted operating costs are treated as fixed,
  spread equally across 12 months.
- 30% are treated as variable with passenger volumes.
- The fixed/variable split is illustrative, not disclosed by Heathrow.
- No revenue-yield growth or cost inflation is assumed.
- Applying 2025 passenger volumes reproduces the 2025 annual
  revenue and adjusted operating-cost benchmarks.

### Modelled 2026 outlook

| Measure | Model output |
|---|---:|
| Revenue | £3,606.95m |
| Adjusted operating costs | £1,586.89m |
| Adjusted EBITDA | £2,020.06m |
| Adjusted EBITDA margin | 56.00% |

All monthly financial figures are modelled estimates.
The reconciliation checks arithmetic, not forecast accuracy.

In [13]:
baseline_passengers = (
    traffic_monthly.loc[
        traffic_monthly["month"].dt.year == 2025
    ]
    .set_index("month")["passengers"]
)

model_2026["baseline_passengers"] = (
    baseline_passengers
    .reindex(model_2026["month"] - pd.DateOffset(years=1))
    .to_numpy()
)

model_2026[
    ["month", "passengers", "baseline_passengers"]
]

,month,passengers,baseline_passengers
0,2026-01-01,6460877,6319076
1,2026-02-01,5819080,5709458
2,2026-03-01,6645184,6218374
3,2026-04-01,6721032,7098983
4,2026-05-01,7121212,7204388
5,2026-06-01,7227494,7359252
6,2026-07-01,7859408,7981137
7,2026-08-01,7866510,8035996
8,2026-09-01,7179707,7347948
9,2026-10-01,7365772,7365772


In [14]:
model_2026["baseline_revenue_gbp_m"] = (
    model_2026["baseline_passengers"]
    * revenue_per_passenger_assumption
    / 1_000_000
)

model_2026["revenue_variance_gbp_m"] = (
    model_2026["modelled_revenue_gbp_m"]
    - model_2026["baseline_revenue_gbp_m"]
)

model_2026["revenue_variance_pct"] = (
    model_2026["revenue_variance_gbp_m"]
    / model_2026["baseline_revenue_gbp_m"]
    * 100
)

model_2026[
    [
        "month",
        "baseline_revenue_gbp_m",
        "modelled_revenue_gbp_m",
        "revenue_variance_gbp_m",
        "revenue_variance_pct"
    ]
].round({
    "baseline_revenue_gbp_m": 2,
    "modelled_revenue_gbp_m": 2,
    "revenue_variance_gbp_m": 2,
    "revenue_variance_pct": 2
})

,month,baseline_revenue_gbp_m,modelled_revenue_gbp_m,revenue_variance_gbp_m,revenue_variance_pct
0,2026-01-01,271.05,277.14,6.08,2.24
1,2026-02-01,244.90,249.61,4.70,1.92
2,2026-03-01,266.73,285.04,18.31,6.86
3,2026-04-01,304.51,288.30,-16.21,-5.32
4,2026-05-01,309.03,305.46,-3.57,-1.15
5,2026-06-01,315.67,310.02,-5.65,-1.79
6,2026-07-01,342.35,337.13,-5.22,-1.53
7,2026-08-01,344.70,337.43,-7.27,-2.11
8,2026-09-01,315.19,307.97,-7.22,-2.29
9,2026-10-01,315.95,315.95,0.00,0.00


In [15]:
model_2026["baseline_operating_cost_gbp_m"] = (
    monthly_fixed_cost_gbp
    + model_2026["baseline_passengers"]
    * variable_cost_per_passenger_gbp
) / 1_000_000

model_2026["cost_variance_gbp_m"] = (
    model_2026["modelled_operating_cost_gbp_m"]
    - model_2026["baseline_operating_cost_gbp_m"]
)

model_2026["cost_variance_pct"] = (
    model_2026["cost_variance_gbp_m"]
    / model_2026["baseline_operating_cost_gbp_m"]
    * 100
)

model_2026[
    [
        "month",
        "baseline_operating_cost_gbp_m",
        "modelled_operating_cost_gbp_m",
        "cost_variance_gbp_m",
        "cost_variance_pct"
    ]
].round({
    "baseline_operating_cost_gbp_m": 2,
    "modelled_operating_cost_gbp_m": 2,
    "cost_variance_gbp_m": 2,
    "cost_variance_pct": 2
})

,month,baseline_operating_cost_gbp_m,modelled_operating_cost_gbp_m,cost_variance_gbp_m,cost_variance_pct
0,2026-01-01,128.36,129.16,0.80,0.62
1,2026-02-01,124.92,125.53,0.62,0.50
2,2026-03-01,127.79,130.20,2.41,1.89
3,2026-04-01,132.76,130.62,-2.13,-1.61
4,2026-05-01,133.35,132.88,-0.47,-0.35
5,2026-06-01,134.23,133.48,-0.74,-0.55
6,2026-07-01,137.74,137.05,-0.69,-0.50
7,2026-08-01,138.05,137.09,-0.96,-0.69
8,2026-09-01,134.16,133.21,-0.95,-0.71
9,2026-10-01,134.26,134.26,0.00,0.00


In [16]:
model_2026["baseline_adjusted_ebitda_gbp_m"] = (
    model_2026["baseline_revenue_gbp_m"]
    - model_2026["baseline_operating_cost_gbp_m"]
)

model_2026["ebitda_variance_gbp_m"] = (
    model_2026["modelled_adjusted_ebitda_gbp_m"]
    - model_2026["baseline_adjusted_ebitda_gbp_m"]
)

model_2026["ebitda_variance_pct"] = (
    model_2026["ebitda_variance_gbp_m"]
    / model_2026["baseline_adjusted_ebitda_gbp_m"]
    * 100
)

model_2026[
    [
        "month",
        "ebitda_variance_gbp_m",
        "ebitda_variance_pct"
    ]
].round({
    "ebitda_variance_gbp_m": 2,
    "ebitda_variance_pct": 2
})

,month,ebitda_variance_gbp_m,ebitda_variance_pct
0,2026-01-01,5.28,3.70
1,2026-02-01,4.08,3.40
2,2026-03-01,15.90,11.44
3,2026-04-01,-14.08,-8.20
4,2026-05-01,-3.10,-1.76
5,2026-06-01,-4.91,-2.70
6,2026-07-01,-4.53,-2.22
7,2026-08-01,-6.31,-3.06
8,2026-09-01,-6.27,-3.46
9,2026-10-01,-0.00,-0.00


In [17]:
model_2026["ebitda_variance_status"] = (
    model_2026["ebitda_variance_gbp_m"]
    .apply(
        lambda value:
        "On baseline" if abs(value) < 0.000001
        else "Favourable" if value > 0
        else "Unfavourable"
    )
)

model_2026[
    ["month", "ebitda_variance_gbp_m", "ebitda_variance_status"]
].round({"ebitda_variance_gbp_m": 2})

,month,ebitda_variance_gbp_m,ebitda_variance_status
0,2026-01-01,5.28,Favourable
1,2026-02-01,4.08,Favourable
2,2026-03-01,15.90,Favourable
3,2026-04-01,-14.08,Unfavourable
4,2026-05-01,-3.10,Unfavourable
5,2026-06-01,-4.91,Unfavourable
6,2026-07-01,-4.53,Unfavourable
7,2026-08-01,-6.31,Unfavourable
8,2026-09-01,-6.27,Unfavourable
9,2026-10-01,-0.00,On baseline


In [18]:
ytd = model_2026.loc[
    model_2026["month"] <= "2026-09-01"
]

ytd_modelled_ebitda = ytd["modelled_adjusted_ebitda_gbp_m"].sum()
ytd_baseline_ebitda = ytd["baseline_adjusted_ebitda_gbp_m"].sum()

ytd_ebitda_variance = ytd_modelled_ebitda - ytd_baseline_ebitda
ytd_ebitda_variance_pct = (
    ytd_ebitda_variance / ytd_baseline_ebitda * 100
)

print(f"Jan–Sep modelled EBITDA: £{ytd_modelled_ebitda:,.2f}m")
print(f"Jan–Sep baseline EBITDA: £{ytd_baseline_ebitda:,.2f}m")
print(f"YTD variance: £{ytd_ebitda_variance:,.2f}m")
print(f"YTD variance (%): {ytd_ebitda_variance_pct:.2f}%")

Jan–Sep modelled EBITDA: £1,508.86m
Jan–Sep baseline EBITDA: £1,522.79m
YTD variance: £-13.94m
YTD variance (%): -0.92%


In [19]:
ytd_revenue_variance = ytd["revenue_variance_gbp_m"].sum()
ytd_cost_variance = ytd["cost_variance_gbp_m"].sum()

driver_check = (
    ytd_revenue_variance
    - ytd_cost_variance
    - ytd_ebitda_variance
)

print(f"Revenue variance: £{ytd_revenue_variance:,.2f}m")
print(f"Cost variance: £{ytd_cost_variance:,.2f}m")
print(f"EBITDA variance: £{ytd_ebitda_variance:,.2f}m")
print(f"Reconciliation difference: £{driver_check:.8f}m")

Revenue variance: £-16.05m
Cost variance: £-2.11m
EBITDA variance: £-13.94m
Reconciliation difference: £0.00000000m


In [20]:
model_2026["financial_data_status"] = "modelled_estimate"
model_2026["baseline_type"] = "2025_monthly_passengers_at_2025_rates"

model_2026.to_csv(
    PROCESSED_DIR / "heathrow_financial_model_2026.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved: heathrow_financial_model_2026.csv")
print("Rows:", len(model_2026))

Saved: heathrow_financial_model_2026.csv
Rows: 12


### Variance analysis against the model baseline

The baseline uses each month's 2025 passenger count with the same
revenue yield and cost assumptions used in the 2026 model.
It is not Heathrow's approved budget.

January–September 2026 modelled results versus baseline:
- Revenue: £16.05m lower.
- Operating costs: £2.11m lower.
- Adjusted EBITDA: £13.94m lower, an unfavourable variance of 0.92%.

The revenue shortfall is partly offset by lower variable costs.
These changes reflect passenger volumes, not measured price changes
or demonstrated cost efficiencies.

October–December variances are zero by construction because the
seasonal naïve forecast repeats the baseline passenger counts.

In [21]:
scenario_assumptions = pd.DataFrame({
    "scenario": ["Baseline", "Upside", "Downside"],
    "q4_passenger_change_pct": [0.0, 5.0, -5.0]
})

scenario_assumptions

,scenario,q4_passenger_change_pct
0,Baseline,0.0
1,Upside,5.0
2,Downside,-5.0


In [22]:
scenario_monthly = model_2026[
    ["month", "passengers", "passenger_status"]
].merge(scenario_assumptions, how="cross")

scenario_monthly["scenario_passengers"] = (
    scenario_monthly["passengers"].astype(float)
)

q4_mask = scenario_monthly["passenger_status"] == "model_forecast"

scenario_monthly.loc[q4_mask, "scenario_passengers"] *= (
    1
    + scenario_monthly.loc[q4_mask, "q4_passenger_change_pct"] / 100
)

scenario_monthly.loc[
    q4_mask,
    ["month", "scenario", "scenario_passengers"]
].round({"scenario_passengers": 0})

,month,scenario,scenario_passengers
27,2026-10-01,Baseline,7365772.0
28,2026-10-01,Upside,7734061.0
29,2026-10-01,Downside,6997483.0
30,2026-11-01,Baseline,6634019.0
31,2026-11-01,Upside,6965720.0
32,2026-11-01,Downside,6302318.0
33,2026-12-01,Baseline,7188658.0
34,2026-12-01,Upside,7548091.0
35,2026-12-01,Downside,6829225.0


In [23]:
scenario_monthly["modelled_revenue_gbp_m"] = (
    scenario_monthly["scenario_passengers"]
    * revenue_per_passenger_assumption
    / 1_000_000
)

scenario_monthly["modelled_operating_cost_gbp_m"] = (
    monthly_fixed_cost_gbp
    + scenario_monthly["scenario_passengers"]
    * variable_cost_per_passenger_gbp
) / 1_000_000

scenario_monthly["modelled_adjusted_ebitda_gbp_m"] = (
    scenario_monthly["modelled_revenue_gbp_m"]
    - scenario_monthly["modelled_operating_cost_gbp_m"]
)

print("Monthly scenario rows:", len(scenario_monthly))

Monthly scenario rows: 36


In [24]:
scenario_summary = (
    scenario_monthly.groupby("scenario", as_index=False)
    .agg(
        passengers=("scenario_passengers", "sum"),
        revenue_gbp_m=("modelled_revenue_gbp_m", "sum"),
        operating_cost_gbp_m=("modelled_operating_cost_gbp_m", "sum"),
        adjusted_ebitda_gbp_m=("modelled_adjusted_ebitda_gbp_m", "sum")
    )
)

scenario_summary["adjusted_ebitda_margin_pct"] = (
    scenario_summary["adjusted_ebitda_gbp_m"]
    / scenario_summary["revenue_gbp_m"]
    * 100
)

scenario_summary.round({
    "passengers": 0,
    "revenue_gbp_m": 2,
    "operating_cost_gbp_m": 2,
    "adjusted_ebitda_gbp_m": 2,
    "adjusted_ebitda_margin_pct": 2
})

,scenario,passengers,revenue_gbp_m,operating_cost_gbp_m,adjusted_ebitda_gbp_m,adjusted_ebitda_margin_pct
0,Baseline,84088953.0,3606.95,1586.89,2020.06,56.00
1,Downside,83029531.0,3561.51,1580.91,1980.60,55.61
2,Upside,85148375.0,3652.40,1592.87,2059.53,56.39


In [25]:
baseline_scenario = (
    scenario_summary.set_index("scenario").loc["Baseline"]
)

print(
    "Revenue difference (£m):",
    round(baseline_scenario["revenue_gbp_m"] - annual_revenue, 8)
)

print(
    "Operating cost difference (£m):",
    round(baseline_scenario["operating_cost_gbp_m"] - annual_costs, 8)
)

print(
    "EBITDA difference (£m):",
    round(baseline_scenario["adjusted_ebitda_gbp_m"] - annual_ebitda, 8)
)

Revenue difference (£m): 0.0
Operating cost difference (£m): -0.0
EBITDA difference (£m): 0.0


In [28]:
scenario_monthly["financial_data_status"] = "modelled_estimate"
scenario_monthly["forecast_as_of"] = "2026-09-30"

scenario_monthly.to_csv(
    PROCESSED_DIR / "heathrow_scenarios_monthly_2026.csv",
    index=False,
    date_format="%Y-%m-%d"
)

scenario_summary.to_csv(
    PROCESSED_DIR / "heathrow_scenario_summary_2026.csv",
    index=False
)

print("Saved monthly scenarios and annual scenario summary.")

Saved monthly scenarios and annual scenario summary.


### Passenger-demand scenarios

Scenarios adjust October–December forecast passengers only:
- Baseline: no adjustment.
- Upside: 5% above the Q4 passenger forecast.
- Downside: 5% below the Q4 passenger forecast.

January–September reported passengers remain unchanged.
Revenue yield, fixed costs and variable cost per passenger remain unchanged.

| Scenario | Full-year revenue (£m) | Adjusted EBITDA (£m) | EBITDA margin |
|---|---:|---:|---:|
| Baseline | 3,606.95 | 2,020.06 | 56.00% |
| Upside | 3,652.40 | 2,059.53 | 56.39% |
| Downside | 3,561.51 | 1,980.60 | 55.61% |

A 5% change in Q4 passengers changes annual modelled EBITDA
by approximately £39.46m.

These are illustrative sensitivities, not statistical prediction intervals.
The Baseline scenario reconciles to the original financial model.